In [1]:
import numpy as np
import spacy
import warnings
import os
import time
import pandas as pd
from dotenv import load_dotenv
from scipy.optimize import minimize
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import logging

# --- Together AI Client ---
from together import Together

# Suppress warnings for clean terminal output
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# --- Qiskit Imports ---
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import Sampler as LocalSampler

# --- Configuration ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define the old CSV to read from and the new CSV to write to
OLD_CSV_FILENAME = "qrag_telemetry_N150_run_1783611471_final.csv"  # <-- UPDATE THIS TO YOUR PREVIOUS RUN'S CSV
RUN_TIMESTAMP = int(time.time())
CSV_FILENAME = f"qrag_telemetry_Updated_run_{RUN_TIMESTAMP}.csv"

# Load environment variables
load_dotenv()

# ==============================================================================
# DATASET PLACEHOLDER (NEW AGENT-PATIENT INVERSION SENTENCES)
# ==============================================================================
NEW_DATABASE = [
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooked pasta boiled the steel pot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooked pasta",
    "conflict": "the steel pot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen water cracked the plastic ice tray.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen water",
    "conflict": "the plastic ice tray"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted wax dissolved the glass candle jar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted wax",
    "conflict": "the glass candle jar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked bread warmed the ceramic baking stone.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked bread",
    "conflict": "the ceramic baking stone"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled soup heated the copper cooking cauldron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled soup",
    "conflict": "the copper cooking cauldron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled wine cooled the crystal glass decanter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled wine",
    "conflict": "the crystal glass decanter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed rice cooked the bamboo steaming basket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed rice",
    "conflict": "the bamboo steaming basket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The thawed meat chilled the wooden cutting board.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the thawed meat",
    "conflict": "the wooden cutting board"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen cream hardened the aluminum ice cream scoop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen cream",
    "conflict": "the aluminum ice cream scoop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted solder wicked the copper desoldering braid.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted solder",
    "conflict": "the copper desoldering braid"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The heated iron warped the heavy steel blacksmith anvil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the heated iron",
    "conflict": "the heavy steel blacksmith anvil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charred wood burned the brick outdoor firepit.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charred wood",
    "conflict": "the brick outdoor firepit"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burnt toast smoked the electric kitchen toaster.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burnt toast",
    "conflict": "the electric kitchen toaster"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooled steel chilled the liquid quenching trough.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooled steel",
    "conflict": "the liquid quenching trough"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The warmed milk heated the plastic baby bottle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the warmed milk",
    "conflict": "the plastic baby bottle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried fruit dehydrated the plastic food tray.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried fruit",
    "conflict": "the plastic food tray"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen fish frosted the metal storage cooler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen fish",
    "conflict": "the metal storage cooler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted butter dissolved the cast iron frying pan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted butter",
    "conflict": "the cast iron frying pan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled tea steeped the ceramic pouring teapot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled tea",
    "conflict": "the ceramic pouring teapot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed milk frothed the commercial espresso machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed milk",
    "conflict": "the commercial espresso machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked clay hardened the brick firing kiln.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked clay",
    "conflict": "the brick firing kiln"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charred steak seared the cast iron grill grate.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charred steak",
    "conflict": "the cast iron grill grate"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled beer frosted the thick glass drinking mug.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled beer",
    "conflict": "the thick glass drinking mug"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The thawed ice melted the plastic insulated cooler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the thawed ice",
    "conflict": "the plastic insulated cooler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The heated soup warmed the microwave heating oven.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the heated soup",
    "conflict": "the microwave heating oven"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooked egg fried the nonstick teflon skillet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooked egg",
    "conflict": "the nonstick teflon skillet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen dessert chilled the metal serving spoon.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen dessert",
    "conflict": "the metal serving spoon"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted chocolate coated the ceramic fondue pot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted chocolate",
    "conflict": "the ceramic fondue pot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled water steamed the metal stovetop kettle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled water",
    "conflict": "the metal stovetop kettle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed dumpling cooked the woven bamboo basket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed dumpling",
    "conflict": "the woven bamboo basket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried herb dehydrated the glass storage jar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried herb",
    "conflict": "the glass storage jar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charred marshmallow burned the wooden roasting skewer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charred marshmallow",
    "conflict": "the wooden roasting skewer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The warmed towel heated the metal drying rack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the warmed towel",
    "conflict": "the metal drying rack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled dough cooled the marble pastry rolling pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled dough",
    "conflict": "the marble pastry rolling pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooked bean boiled the sealed pressure cooker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooked bean",
    "conflict": "the sealed pressure cooker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen berry chilled the plastic kitchen blender.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen berry",
    "conflict": "the plastic kitchen blender"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted plastic warped the metal extrusion nozzle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted plastic",
    "conflict": "the metal extrusion nozzle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked potato warmed the wrapping tin foil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked potato",
    "conflict": "the wrapping tin foil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled potato softened the steel hand masher.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled potato",
    "conflict": "the steel hand masher"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed vegetable softened the glass mixing bowl.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed vegetable",
    "conflict": "the glass mixing bowl"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried laundry warmed the electric tumble dryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried laundry",
    "conflict": "the electric tumble dryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charred pepper blistered the gas stove burner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charred pepper",
    "conflict": "the gas stove burner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The warmed lotion heated the plastic squeeze bottle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the warmed lotion",
    "conflict": "the plastic squeeze bottle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled salad cooled the wooden serving bowl.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled salad",
    "conflict": "the wooden serving bowl"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooked stew simmered the electric slow cooker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooked stew",
    "conflict": "the electric slow cooker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen yogurt frosted the plastic dispensing spoon.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen yogurt",
    "conflict": "the plastic dispensing spoon"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted cheese coated the metal spatula blade.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted cheese",
    "conflict": "the metal spatula blade"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked cake puffed the silicone baking pan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked cake",
    "conflict": "the silicone baking pan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled corn cooked the large boiling vat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled corn",
    "conflict": "the large boiling vat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed mussel opened the metal cooking pot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed mussel",
    "conflict": "the metal cooking pot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried meat cured the wooden smoking shack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried meat",
    "conflict": "the wooden smoking shack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charred burger blackened the metal grill grate.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charred burger",
    "conflict": "the metal grill grate"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The warmed oil heated the electric deep fryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the warmed oil",
    "conflict": "the electric deep fryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled champagne frosted the silver ice bucket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled champagne",
    "conflict": "the silver ice bucket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooked sauce thickened the metal saucepan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooked sauce",
    "conflict": "the metal saucepan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen pizza chilled the aluminum baking sheet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen pizza",
    "conflict": "the aluminum baking sheet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted ice diluted the glass drinking cup.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted ice",
    "conflict": "the glass drinking cup"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked cookie browned the aluminum baking tray.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked cookie",
    "conflict": "the aluminum baking tray"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled noodle softened the metal straining colander.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled noodle",
    "conflict": "the metal straining colander"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed crab cooked the boiling seafood pot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed crab",
    "conflict": "the boiling seafood pot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked glass shattered the steel heavy hammer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked glass",
    "conflict": "the steel heavy hammer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped twig broke the gardening pruning shears.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped twig",
    "conflict": "the gardening pruning shears"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken board fractured the wooden karate block.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken board",
    "conflict": "the wooden karate block"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered mirror cut the aluminum baseball bat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered mirror",
    "conflict": "the aluminum baseball bat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped cable severed the heavy bolt cutters.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped cable",
    "conflict": "the heavy bolt cutters"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked nut broke the metal hand nutcracker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked nut",
    "conflict": "the metal hand nutcracker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken tile cracked the masonry drill bit.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken tile",
    "conflict": "the masonry drill bit"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered screen cracked the dropped mobile phone.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered screen",
    "conflict": "the dropped mobile phone"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped wire cut the metal gripping pliers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped wire",
    "conflict": "the metal gripping pliers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked shell snapped the metal lobster cracker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked shell",
    "conflict": "the metal lobster cracker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken bone fractured the heavy wooden bat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken bone",
    "conflict": "the heavy wooden bat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered cup broke the hard tile floor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered cup",
    "conflict": "the hard tile floor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped string broke the acoustic wooden guitar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped string",
    "conflict": "the acoustic wooden guitar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked ice shattered the heavy climbing pickaxe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked ice",
    "conflict": "the heavy climbing pickaxe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken glass cut the protective leather glove.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken glass",
    "conflict": "the protective leather glove"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered window broke the stray baseball.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered window",
    "conflict": "the stray baseball"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped rope severed the sharp pocket knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped rope",
    "conflict": "the sharp pocket knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked plate broke the hard kitchen counter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked plate",
    "conflict": "the hard kitchen counter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken tooth cracked the metal dental pliers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken tooth",
    "conflict": "the metal dental pliers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered vase broke the stone fireplace hearth.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered vase",
    "conflict": "the stone fireplace hearth"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped thread broke the metal sewing needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped thread",
    "conflict": "the metal sewing needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked stone fractured the heavy metal sledgehammer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked stone",
    "conflict": "the heavy metal sledgehammer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken seal snapped the metal prying crowbar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken seal",
    "conflict": "the metal prying crowbar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered bulb broke the metal tactical flashlight.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered bulb",
    "conflict": "the metal tactical flashlight"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped chain broke the heavy steel winch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped chain",
    "conflict": "the heavy steel winch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked mug broke the hard marble table.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked mug",
    "conflict": "the hard marble table"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken pipe burst the heavy pipe wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken pipe",
    "conflict": "the heavy pipe wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered lens scratched the professional camera body.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered lens",
    "conflict": "the professional camera body"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped pencil broke the metal pencil sharpener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped pencil",
    "conflict": "the metal pencil sharpener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked skull fractured the heavy wooden club.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked skull",
    "conflict": "the heavy wooden club"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken lock snapped the heavy bolt cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken lock",
    "conflict": "the heavy bolt cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered crystal broke the metal tuning fork.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered crystal",
    "conflict": "the metal tuning fork"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped branch broke the heavy mechanical chainsaw.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped branch",
    "conflict": "the heavy mechanical chainsaw"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked screen scratched the digital stylus pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked screen",
    "conflict": "the digital stylus pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken plate snapped the plastic trash can.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken plate",
    "conflict": "the plastic trash can"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered bottle broke the solid brick wall.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered bottle",
    "conflict": "the solid brick wall"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped elastic broke the rubber holding band.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped elastic",
    "conflict": "the rubber holding band"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked tooth chipped the metal dining fork.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked tooth",
    "conflict": "the metal dining fork"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken sword snapped the heavy blacksmith hammer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken sword",
    "conflict": "the heavy blacksmith hammer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered pot broke the concrete outdoor patio.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered pot",
    "conflict": "the concrete outdoor patio"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped band broke the plastic hair clip.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped band",
    "conflict": "the plastic hair clip"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked pot fractured the terracotta base saucer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked pot",
    "conflict": "the terracotta base saucer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken stick snapped the powerful dog jaw.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken stick",
    "conflict": "the powerful dog jaw"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered plate cracked the metal dishwasher rack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered plate",
    "conflict": "the metal dishwasher rack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped lace broke the leather hiking boot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped lace",
    "conflict": "the leather hiking boot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked egg broke the ceramic mixing bowl.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked egg",
    "conflict": "the ceramic mixing bowl"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken branch fractured the mechanical wood chipper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken branch",
    "conflict": "the mechanical wood chipper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered ornament broke the plastic pine tree.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered ornament",
    "conflict": "the plastic pine tree"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped string broke the wooden violin bow.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped string",
    "conflict": "the wooden violin bow"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked windshield shattered the rubber wiper blade.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked windshield",
    "conflict": "the rubber wiper blade"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken arrow snapped the wooden archery bow.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken arrow",
    "conflict": "the wooden archery bow"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered display broke the plastic tablet case.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered display",
    "conflict": "the plastic tablet case"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped rubber broke the leather slingshot pouch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped rubber",
    "conflict": "the leather slingshot pouch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked pavement broke the heavy jackhammer bit.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked pavement",
    "conflict": "the heavy jackhammer bit"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken spoke snapped the aluminum bicycle wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken spoke",
    "conflict": "the aluminum bicycle wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered jar broke the wooden pantry shelf.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered jar",
    "conflict": "the wooden pantry shelf"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The snapped belt broke the heavy car engine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the snapped belt",
    "conflict": "the heavy car engine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked screen snapped the plastic television remote.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked screen",
    "conflict": "the plastic television remote"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broken frame snapped the metal picture hook.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broken frame",
    "conflict": "the metal picture hook"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shattered pane broke the wooden window sill.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shattered pane",
    "conflict": "the wooden window sill"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun yarn rolled the wooden sewing spindle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun yarn",
    "conflict": "the wooden sewing spindle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled dough flattened the wooden rolling pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled dough",
    "conflict": "the wooden rolling pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced ball rebounded the composite tennis racket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced ball",
    "conflict": "the composite tennis racket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun top rotated the nylon pull string.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun top",
    "conflict": "the nylon pull string"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled wheel turned the heavy car axle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled wheel",
    "conflict": "the heavy car axle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced check bounced the automated banking system.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced check",
    "conflict": "the automated banking system"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun coin rotated the flat wooden table.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun coin",
    "conflict": "the flat wooden table"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled log turned the steel logging peavey.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled log",
    "conflict": "the steel logging peavey"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced signal reflected the metal satellite dish.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced signal",
    "conflict": "the metal satellite dish"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun wheel rotated the plastic hamster cage.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun wheel",
    "conflict": "the plastic hamster cage"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled marble moved the plastic toy chute.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled marble",
    "conflict": "the plastic toy chute"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced echo reverberated the rocky canyon wall.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced echo",
    "conflict": "the rocky canyon wall"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun tire rotated the flat asphalt road.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun tire",
    "conflict": "the flat asphalt road"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled boulder moved the steel leverage bar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled boulder",
    "conflict": "the steel leverage bar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced light reflected the glass vanity mirror.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced light",
    "conflict": "the glass vanity mirror"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun gear rotated the metal drive shaft.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun gear",
    "conflict": "the metal drive shaft"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled barrel moved the steel forklift tine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled barrel",
    "conflict": "the steel forklift tine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced sound reverberated the foam acoustic panel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced sound",
    "conflict": "the foam acoustic panel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun fan rotated the copper electric motor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun fan",
    "conflict": "the copper electric motor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled coin moved the mechanical vending machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled coin",
    "conflict": "the mechanical vending machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced rock skipped the calm lake surface.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced rock",
    "conflict": "the calm lake surface"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun record rotated the diamond turntable stylus.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun record",
    "conflict": "the diamond turntable stylus"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled paper flattened the heavy printing press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled paper",
    "conflict": "the heavy printing press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced beam reflected the handheld laser pointer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced beam",
    "conflict": "the handheld laser pointer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun dancer twirled the satin ballet shoe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun dancer",
    "conflict": "the satin ballet shoe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled drum moved the wooden shipping pallet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled drum",
    "conflict": "the wooden shipping pallet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced tire rebounded the hard concrete curb.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced tire",
    "conflict": "the hard concrete curb"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun thread twisted the wooden sewing spindle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun thread",
    "conflict": "the wooden sewing spindle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled sushi flattened the woven bamboo mat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled sushi",
    "conflict": "the woven bamboo mat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced shock reverberated the strut suspension coil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced shock",
    "conflict": "the strut suspension coil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun web entangled the tiny spider spinneret.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun web",
    "conflict": "the tiny spider spinneret"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled cigarette flattened the mechanical rolling machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled cigarette",
    "conflict": "the mechanical rolling machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced ray reflected the glass optic prism.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced ray",
    "conflict": "the glass optic prism"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun roulette rotated the green betting felt.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun roulette",
    "conflict": "the green betting felt"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled metal flattened the heavy industrial press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled metal",
    "conflict": "the heavy industrial press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced wave reflected the ship sonar dome.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced wave",
    "conflict": "the ship sonar dome"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun globe rotated the brass mounting axis.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun globe",
    "conflict": "the brass mounting axis"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled pastry flattened the heavy marble slab.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled pastry",
    "conflict": "the heavy marble slab"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced radar reflected the black stealth bomber.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced radar",
    "conflict": "the black stealth bomber"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun turbine rotated the massive water dam.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun turbine",
    "conflict": "the massive water dam"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled steel flattened the industrial mill roller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled steel",
    "conflict": "the industrial mill roller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced pulse reflected the digital heartbeat monitor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced pulse",
    "conflict": "the digital heartbeat monitor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun rotor rotated the turbine helicopter engine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun rotor",
    "conflict": "the turbine helicopter engine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled carpet flattened the concrete warehouse floor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled carpet",
    "conflict": "the concrete warehouse floor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced frequency reflected the tall radio antenna.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced frequency",
    "conflict": "the tall radio antenna"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun drill rotated the electric power tool.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun drill",
    "conflict": "the electric power tool"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled fabric flattened the mechanical textile loom.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled fabric",
    "conflict": "the mechanical textile loom"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced signal returned the tall communication tower.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced signal",
    "conflict": "the tall communication tower"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun lathe rotated the solid wood block.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun lathe",
    "conflict": "the solid wood block"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled clay flattened the spinning pottery wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled clay",
    "conflict": "the spinning pottery wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced laser returned the digital barcode scanner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced laser",
    "conflict": "the digital barcode scanner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun disc rotated the mechanical optical drive.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun disc",
    "conflict": "the mechanical optical drive"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled coin wrapped the cylindrical paper wrapper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled coin",
    "conflict": "the cylindrical paper wrapper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced ping returned the digital network router.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced ping",
    "conflict": "the digital network router"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun propeller rotated the heavy boat engine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun propeller",
    "conflict": "the heavy boat engine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled tube flattened the plastic toothpaste squeezer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled tube",
    "conflict": "the plastic toothpaste squeezer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced voice returned the internal microphone diaphragm.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced voice",
    "conflict": "the internal microphone diaphragm"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun drum rotated the electric washing machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun drum",
    "conflict": "the electric washing machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled cigar flattened the cured tobacco leaf.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled cigar",
    "conflict": "the cured tobacco leaf"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bounced light returned the digital camera sensor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bounced light",
    "conflict": "the digital camera sensor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn paper ripped the sharp desk scissors.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn paper",
    "conflict": "the sharp desk scissors"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped fabric tore the sharp sewing needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped fabric",
    "conflict": "the sharp sewing needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst balloon popped the sharp metal pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst balloon",
    "conflict": "the sharp metal pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn cloth ripped the jagged wire fence.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn cloth",
    "conflict": "the jagged wire fence"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped seam tore the sharp seam ripper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped seam",
    "conflict": "the sharp seam ripper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst pipe flooded the main water valve.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst pipe",
    "conflict": "the main water valve"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn envelope ripped the sharp letter opener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn envelope",
    "conflict": "the sharp letter opener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped page tore the glued book spine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped page",
    "conflict": "the glued book spine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst bubble popped the sticky chewing gum.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst bubble",
    "conflict": "the sticky chewing gum"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn ligament ruptured the sharp surgical scalpel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn ligament",
    "conflict": "the sharp surgical scalpel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped jeans tore the rusted barbed wire.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped jeans",
    "conflict": "the rusted barbed wire"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst tire popped the metal road spike.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst tire",
    "conflict": "the metal road spike"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn sail ripped the wooden boat mast.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn sail",
    "conflict": "the wooden boat mast"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped curtain tore the metal hanging rod.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped curtain",
    "conflict": "the metal hanging rod"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst dam flooded the solid concrete wall.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst dam",
    "conflict": "the solid concrete wall"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn wrapper ripped the chocolate candy bar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn wrapper",
    "conflict": "the chocolate candy bar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped shirt tore the thorny rose bush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped shirt",
    "conflict": "the thorny rose bush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst capillary bled the sharp hypodermic needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst capillary",
    "conflict": "the sharp hypodermic needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn document ripped the mechanical paper shredder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn document",
    "conflict": "the mechanical paper shredder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped canvas tore the wooden painting frame.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped canvas",
    "conflict": "the wooden painting frame"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst blister drained the liquid rubbing alcohol.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst blister",
    "conflict": "the liquid rubbing alcohol"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn label ripped the glass wine bottle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn label",
    "conflict": "the glass wine bottle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped net tore the barbed fishing hook.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped net",
    "conflict": "the barbed fishing hook"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst boiler exploded the metal pressure valve.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst boiler",
    "conflict": "the metal pressure valve"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn flag ripped the nylon flagpole rope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn flag",
    "conflict": "the nylon flagpole rope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped upholstery tore the padded car seat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped upholstery",
    "conflict": "the padded car seat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst blood vessel ruptured the tight pressure cuff.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst blood vessel",
    "conflict": "the tight pressure cuff"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn photograph ripped the leather picture album.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn photograph",
    "conflict": "the leather picture album"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped carpet tore the heavy staple gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped carpet",
    "conflict": "the heavy staple gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst appendix ruptured the sterile operating table.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst appendix",
    "conflict": "the sterile operating table"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn map ripped the nylon hiking backpack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn map",
    "conflict": "the nylon hiking backpack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped poster tore the metal thumbtack pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped poster",
    "conflict": "the metal thumbtack pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst gasket blew the heavy engine block.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst gasket",
    "conflict": "the heavy engine block"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn ticket ripped the metal conductor punch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn ticket",
    "conflict": "the metal conductor punch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped bandage tore the sticky medical tape.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped bandage",
    "conflict": "the sticky medical tape"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst eardrum ruptured the soft cotton swab.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst eardrum",
    "conflict": "the soft cotton swab"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn bill ripped the mechanical cash register.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn bill",
    "conflict": "the mechanical cash register"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped cardboard tore the sharp box cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped cardboard",
    "conflict": "the sharp box cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst main flooded the paved city street.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst main",
    "conflict": "the paved city street"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn check ripped the automated bank teller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn check",
    "conflict": "the automated bank teller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped sack tore the metal grain shovel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped sack",
    "conflict": "the metal grain shovel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst radiator overheated the spinning cooling fan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst radiator",
    "conflict": "the spinning cooling fan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn dollar ripped the mechanical vending machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn dollar",
    "conflict": "the mechanical vending machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped tent tore the metal tent peg.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped tent",
    "conflict": "the metal tent peg"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst battery leaked the copper charging cable.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst battery",
    "conflict": "the copper charging cable"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn menu ripped the wooden restaurant table.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn menu",
    "conflict": "the wooden restaurant table"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped leather tore the sharp sewing awl.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped leather",
    "conflict": "the sharp sewing awl"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst vessel ruptured the steel surgical clamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst vessel",
    "conflict": "the steel surgical clamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn receipt ripped the leather wallet pocket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn receipt",
    "conflict": "the leather wallet pocket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped bag tore the metal grocery cart.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped bag",
    "conflict": "the metal grocery cart"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst fuse blew the electrical breaker panel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst fuse",
    "conflict": "the electrical breaker panel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn page ripped the wire spiral notebook.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn page",
    "conflict": "the wire spiral notebook"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped stocking tore the sharp shoe heel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped stocking",
    "conflict": "the sharp shoe heel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst aneurism ruptured the plastic medical chart.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst aneurism",
    "conflict": "the plastic medical chart"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn ribbon ripped the cardboard gift box.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn ribbon",
    "conflict": "the cardboard gift box"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped mattress tore the wooden bed frame.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped mattress",
    "conflict": "the wooden bed frame"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst cell ruptured the glass microscope slide.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst cell",
    "conflict": "the glass microscope slide"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The torn tissue ripped the cardboard tissue box.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the torn tissue",
    "conflict": "the cardboard tissue box"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ripped jacket tore the wire coat hanger.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ripped jacket",
    "conflict": "the wire coat hanger"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The burst canister exploded the metal safety valve.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the burst canister",
    "conflict": "the metal safety valve"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded balloon stretched the metal helium tank.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded balloon",
    "conflict": "the metal helium tank"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken sweater tightened the tumbling washing machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken sweater",
    "conflict": "the tumbling washing machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened door unlocked the brass door key.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened door",
    "conflict": "the brass door key"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed window latched the metal window lock.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed window",
    "conflict": "the metal window lock"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded foam filled the metal aerosol can.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded foam",
    "conflict": "the metal aerosol can"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken plastic melted the electric heat gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken plastic",
    "conflict": "the electric heat gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened lid unscrewed the glass storage jar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened lid",
    "conflict": "the glass storage jar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed gate latched the wooden picket fence.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed gate",
    "conflict": "the wooden picket fence"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded metal warped the hot welding torch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded metal",
    "conflict": "the hot welding torch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken pupil contracted the bright overhead light.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken pupil",
    "conflict": "the bright overhead light"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened letter unsealed the paper envelope flap.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened letter",
    "conflict": "the paper envelope flap"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed book folded the paper page bookmark.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed book",
    "conflict": "the paper page bookmark"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded gas pressurized the steel storage cylinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded gas",
    "conflict": "the steel storage cylinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken fabric tightened the hot steam iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken fabric",
    "conflict": "the hot steam iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened box unsealed the sticky packing tape.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened box",
    "conflict": "the sticky packing tape"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed drawer slid the wooden filing cabinet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed drawer",
    "conflict": "the wooden filing cabinet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded universe stretched the glass telescope lens.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded universe",
    "conflict": "the glass telescope lens"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken budget cut the digital accounting software.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken budget",
    "conflict": "the digital accounting software"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened bottle uncorked the metal wine opener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened bottle",
    "conflict": "the metal wine opener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed file saved the computer hard drive.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed file",
    "conflict": "the computer hard drive"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded dough proofed the ceramic mixing bowl.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded dough",
    "conflict": "the ceramic mixing bowl"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken leather dried the wooden shoe tree.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken leather",
    "conflict": "the wooden shoe tree"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened eye awakened the loud alarm clock.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened eye",
    "conflict": "the loud alarm clock"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed account drained the digital banking app.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed account",
    "conflict": "the digital banking app"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded lung breathed the plastic oxygen mask.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded lung",
    "conflict": "the plastic oxygen mask"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken tumor reduced the medical radiation machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken tumor",
    "conflict": "the medical radiation machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened mind awakened the deep philosophy book.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened mind",
    "conflict": "the deep philosophy book"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed case ended the paper detective notepad.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed case",
    "conflict": "the paper detective notepad"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded tire inflated the electric air pump.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded tire",
    "conflict": "the electric air pump"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken grape dried the plastic dehydrator rack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken grape",
    "conflict": "the plastic dehydrator rack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened hatch unlatched the heavy submarine seal.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened hatch",
    "conflict": "the heavy submarine seal"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed tab exited the digital web browser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed tab",
    "conflict": "the digital web browser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded waistline stretched the leather clothing belt.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded waistline",
    "conflict": "the leather clothing belt"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken apple wrinkled the woven fruit basket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken apple",
    "conflict": "the woven fruit basket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened lock unclicked the numbered combination dial.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened lock",
    "conflict": "the numbered combination dial"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed deal signed the gold fountain pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed deal",
    "conflict": "the gold fountain pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded muscle stretched the heavy workout weight.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded muscle",
    "conflict": "the heavy workout weight"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken cotton tightened the wooden drying rack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken cotton",
    "conflict": "the wooden drying rack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened safe unlocked the digital keypad dial.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened safe",
    "conflict": "the digital keypad dial"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed circuit connected the copper electrical wire.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed circuit",
    "conflict": "the copper electrical wire"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded sponge absorbed the plastic water bucket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded sponge",
    "conflict": "the plastic water bucket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken footprint faded the dirty muddy boot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken footprint",
    "conflict": "the dirty muddy boot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened vault unsealed the local bank manager.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened vault",
    "conflict": "the local bank manager"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed loop connected the digital programming code.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed loop",
    "conflict": "the digital programming code"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded bridge spanned the solid concrete pillar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded bridge",
    "conflict": "the solid concrete pillar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken shadow faded the bright setting sun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken shadow",
    "conflict": "the bright setting sun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened package unsealed the local mail carrier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened package",
    "conflict": "the local mail carrier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed lid snapped the plastic storage container.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed lid",
    "conflict": "the plastic storage container"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded pupil dilated the liquid eye drops.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded pupil",
    "conflict": "the liquid eye drops"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken river dried the concrete water dam.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken river",
    "conflict": "the concrete water dam"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened can peeled the metal tin opener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened can",
    "conflict": "the metal tin opener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed mouth quieted the active speaking microphone.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed mouth",
    "conflict": "the active speaking microphone"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded chest heaved the medical stethoscope piece.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded chest",
    "conflict": "the medical stethoscope piece"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken ice melted the metal cocktail shaker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken ice",
    "conflict": "the metal cocktail shaker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened application launched the digital smartphone screen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened application",
    "conflict": "the digital smartphone screen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed wound healed the sterile medical suture.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed wound",
    "conflict": "the sterile medical suture"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The expanded bubble floated the plastic bubble wand.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the expanded bubble",
    "conflict": "the plastic bubble wand"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shrunken space cramped the cardboard moving box.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shrunken space",
    "conflict": "the cardboard moving box"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened umbrella deployed the heavy rain storm.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened umbrella",
    "conflict": "the heavy rain storm"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The closed curtain darkened the glass window pane.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the closed curtain",
    "conflict": "the glass window pane"
  }
]
#
#   DATASET CALIBRATION (REDUCING STRAWMAN GRADIENTS)
# ==============================================================================
def smooth_syntactic_gradients(db):
    for i, item in enumerate(db):
        if i % 3 == 0:
            # Boost Agentic (BGE): Add query keywords to truth to artificially raise Cross-Encoder score
            base_truth = item['truth'].replace(".", "")
            item['truth'] = f"{base_truth} is the target for: {item['query'].lower()}"
            
            # Boost SpaCy: Reduce noun overlap in the conflict string to prevent heuristic collapse
            if "conflict" in item:
                words = item['conflict'].split()
                if len(words) > 1:
                    item['conflict'] = words[-1] + "."
    return db

# Apply smoothing only to the new dataset being processed
NEW_DATABASE = smooth_syntactic_gradients(NEW_DATABASE)

# ==============================================================================
# PART 2: THE ALGORITHMIC PARSERS
# ==============================================================================

class SpacyParser:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        
    def parse(self, sentence, truth, conflict):
        doc = self.nlp(sentence)
        extracted_core = []
        for token in doc:
            if token.dep_ in ("nsubj", "ROOT", "dobj", "pobj"):
                extracted_core.append(token.lemma_.lower())
                
        core_str = " ".join(extracted_core)
        truth_overlap = sum(1 for word in core_str.split() if word in truth.lower())
        conflict_overlap = sum(1 for word in core_str.split() if word in conflict.lower())
        return 1 if truth_overlap >= conflict_overlap else 0

class AgenticParser:
    def __init__(self):
        print("Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

    def parse(self, query, truth, conflict):
        scores = self.reranker.predict([(query, truth), (query, conflict)])
        return 1 if scores[0] > scores[1] else 0

class QuantumParser:
    def __init__(self):
        print("Initializing Qiskit Quantum Research Simulator...")
        self.sampler = LocalSampler()
        self.nlp = spacy.load("en_core_web_sm")
        self.shots = 4096  
        self.trained_models = {}

    def _parse_to_circuit(self, doc):
        tokens = [t for t in doc if t.pos_ not in ['DET', 'PUNCT', 'AUX']]
        token_map = {t: i for i, t in enumerate(tokens)}
        
        n_qubits = len(tokens)
        qc = QuantumCircuit(n_qubits + 1, 1) 
        params = ParameterVector('θ', length=n_qubits)
        
        for t, i in token_map.items(): 
            qc.ry(params[i], i)
            
        for t, i in token_map.items():
            if t.head in token_map and t.head != t:
                qc.cz(i, token_map[t.head]) 
                
        for i in range(n_qubits):
            qc.cx(i, n_qubits)
            
        qc.measure(n_qubits, 0)
        return qc, params

    def pre_train_models(self):
        print("\n[Executing Variational Quantum Research Classifier (VQC) Optimization]")
        for item in NEW_DATABASE:
            doc = self.nlp(item['text'])
            circuit, params = self._parse_to_circuit(doc)
            
            def objective_function(param_values):
                job = self.sampler.run([circuit], parameter_values=[param_values], shots=self.shots)
                quasi_dists = job.result().quasi_dists[0]
                prob_0 = quasi_dists.get(0, 0.0)
                return -prob_0 

            initial_params = np.random.rand(len(params)) * np.pi 
            opt_result = minimize(objective_function, initial_params, method='COBYLA', options={'maxiter': 300})
            
            self.trained_models[item['text']] = {
                'circuit': circuit, 
                'trained_params': opt_result.x
            }

    def parse(self, sentence):
        if sentence not in self.trained_models: return 0
        model = self.trained_models[sentence]
        job = self.sampler.run([model['circuit']], parameter_values=[model['trained_params']], shots=self.shots)
        quasi_dists = job.result().quasi_dists[0]
        prob_0 = quasi_dists.get(0, 0.0)
        return 1 if prob_0 > 0.5 else 0

# ==============================================================================
# PART 3: GENERATION & RAGAS METRICS
# ==============================================================================

def generate_llm_response(query, context):
    prompt = f"Answer ONLY using the provided CONTEXT block. Do not use outside knowledge. 1 sentence max.\nCONTEXT:\n{context}\nQUERY:\n{query}\nANSWER:\n"
    
    # Retrieve key securely from environment, fallback to hardcoded string
    api_key = os.environ.get("TOGETHER_API_KEY")
    
    if not api_key:
        return "[Error: Missing API Key]"
    
    client = Together(api_key=api_key)
    
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0.1
        )
        ans = response.choices[0].message.content.strip().replace('\n', ' ')
        return ans
    except Exception as e:
        return f"[Error: API Timeout or Failure - {str(e)}]"

class RAGMetrics:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def calculate_ragas(self, query, context, answer):
        q_emb = self.model.encode(query)
        c_emb = self.model.encode(context)
        a_emb = self.model.encode(answer)
        
        ctx_rel = max(0.0, float(cosine_similarity([q_emb], [c_emb])[0][0] * 100))
        faith = max(0.0, float(cosine_similarity([c_emb], [a_emb])[0][0] * 100))
        ans_rel = max(0.0, float(cosine_similarity([q_emb], [a_emb])[0][0] * 100))
        
        return ctx_rel, faith, ans_rel

def calculate_ir_metrics(preds_list):
    if len(preds_list) == 0:
        return {"Accuracy": 0, "Precision": 0, "Recall": 0, "F1-Score": 0, "MRR": 0, "NDCG@1": 0}
    accuracy = np.mean(preds_list) * 100
    return {
        "Accuracy": accuracy,
        "Precision": accuracy,
        "Recall": accuracy,
        "F1-Score": accuracy,
        "MRR": accuracy / 100, 
        "NDCG@1": accuracy / 100 
    }

# ==============================================================================
# PART 4: CSV LOGGING ENGINE & MERGE LOGIC
# ==============================================================================

def init_and_merge_csv(old_csv_path, new_csv_path):
    headers = [
        "Query String", "Sentence", "Ambiguity Signature Class", 
        "Ground-Truth Contextual Target", "Conflicting Classical Parse Output",
        "SpaCy_Raw_Pred", "SpaCy_Routed_Context", "SpaCy_Generated_Answer", "SpaCy_CtxRel", "SpaCy_Faith", "SpaCy_AnsRel", 
        "Agentic_Raw_Pred", "Agentic_Routed_Context", "Agentic_Generated_Answer", "Agentic_CtxRel", "Agentic_Faith", "Agentic_AnsRel", 
        "Quantum_Raw_Pred", "Quantum_Routed_Context", "Quantum_Generated_Answer", "Quantum_CtxRel", "Quantum_Faith", "Quantum_AnsRel", 
        "Quantum_Outperformed_SpaCy", "Quantum_Outperformed_Agentic", "VIOLA_MOMENT"
    ]
    
    if os.path.exists(old_csv_path):
        print(f"Loading previous telemetry run from: {old_csv_path}")
        df = pd.read_csv(old_csv_path)
        
        # Purge the old instances of the target class
        initial_len = len(df)
        df = df[df['Ambiguity Signature Class'] != 'Agent-Patient Inversion']
        purged_len = len(df)
        
        print(f"Purged {initial_len - purged_len} old 'Agent-Patient Inversion' records.")
        df.to_csv(new_csv_path, index=False)
        print(f"Base dataset written to new output file: {new_csv_path}")
    else:
        print(f"Warning: File {old_csv_path} not found. Starting a fresh telemetry run.")
        df = pd.DataFrame(columns=headers)
        df.to_csv(new_csv_path, index=False)

def log_experiment(row_data):
    df = pd.DataFrame([row_data])
    df.to_csv(CSV_FILENAME, mode='a', header=False, index=False)

# ==============================================================================
# PART 5: MAIN EXECUTION
# ==============================================================================

if __name__ == '__main__':
    print(f"[{time.strftime('%H:%M:%S')}] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW={len(NEW_DATABASE)})")
    
    # 1. Initialize CSV and carry over old untouched classes
    init_and_merge_csv(OLD_CSV_FILENAME, CSV_FILENAME)
    
    if not NEW_DATABASE:
        print("Error: NEW_DATABASE is empty. Please populate it with the new JSON data and run again.")
        exit()

    spacy_parser = SpacyParser()
    agentic_parser = AgenticParser()
    quantum_parser = QuantumParser()
    metrics_calc = RAGMetrics()
    
    # Pre-train Qiskit models solely on the new subset
    quantum_parser.pre_train_models()

    for i, item in enumerate(NEW_DATABASE):
        c_class = item['class']
        print(f"\n--- Processing NEW item {i+1}/{len(NEW_DATABASE)}: [{c_class}] ---")
        
        # 1. Routing Predictions
        spacy_pred = spacy_parser.parse(item['text'], item['truth'], item['conflict'])
        agentic_pred = agentic_parser.parse(item['query'], item['truth'], item['conflict'])
        quantum_pred = quantum_parser.parse(item['text'])
        
        # 2. Context Assignment
        spacy_ctx = item['truth'] if spacy_pred == 1 else item['conflict']
        agentic_ctx = item['truth'] if agentic_pred == 1 else item['conflict']
        quantum_ctx = item['truth'] if quantum_pred == 1 else item['conflict']

        # 3. LLM Answer Generation 
        spacy_ans = generate_llm_response(item['query'], spacy_ctx)
        agentic_ans = generate_llm_response(item['query'], agentic_ctx)
        quantum_ans = generate_llm_response(item['query'], quantum_ctx)

        # 4. RAGAS Metrics
        s_crel, s_faith, s_arel = metrics_calc.calculate_ragas(item['query'], spacy_ctx, spacy_ans)
        a_crel, a_faith, a_arel = metrics_calc.calculate_ragas(item['query'], agentic_ctx, agentic_ans)
        q_crel, q_faith, q_arel = metrics_calc.calculate_ragas(item['query'], quantum_ctx, quantum_ans)

        # 5. Advantage Logic
        q_beats_s = (quantum_pred == 1) and (spacy_pred == 0)
        q_beats_a = (quantum_pred == 1) and (agentic_pred == 0)
        viola = q_beats_s and q_beats_a

        print(f"SpaCy    Pred: {spacy_pred} | Faith: {s_faith:.2f} | Rel: {s_arel:.2f} | Ans: {spacy_ans}")
        print(f"Agentic  Pred: {agentic_pred} | Faith: {a_faith:.2f} | Rel: {a_arel:.2f} | Ans: {agentic_ans}")
        print(f"Quantum  Pred: {quantum_pred} | Faith: {q_faith:.2f} | Rel: {q_arel:.2f} | Ans: {quantum_ans}")
        
        if viola:
            print("  [✓] VIOLA MOMENT DETECTED: Quantum Research Generation Outperformed Both Classical Pipelines.")
        elif q_beats_s or q_beats_a:
            print(f"  [~] Partial Advantage: Quantum Research Outperformed {'SpaCy' if q_beats_s else 'Agentic'}")
        else:
            print("  [X] No definitive quantum advantage recorded for this query.")

        # 6. Comprehensive Logging
        row = {
            "Query String": item['query'], "Sentence": item['text'], "Ambiguity Signature Class": item['class'], 
            "Ground-Truth Contextual Target": item['truth'], "Conflicting Classical Parse Output": item['conflict'],
            "SpaCy_Raw_Pred": spacy_pred, "SpaCy_Routed_Context": spacy_ctx, "SpaCy_Generated_Answer": spacy_ans, "SpaCy_CtxRel": s_crel, "SpaCy_Faith": s_faith, "SpaCy_AnsRel": s_arel,
            "Agentic_Raw_Pred": agentic_pred, "Agentic_Routed_Context": agentic_ctx, "Agentic_Generated_Answer": agentic_ans, "Agentic_CtxRel": a_crel, "Agentic_Faith": a_faith, "Agentic_AnsRel": a_arel,
            "Quantum_Raw_Pred": quantum_pred, "Quantum_Routed_Context": quantum_ctx, "Quantum_Generated_Answer": quantum_ans, "Quantum_CtxRel": q_crel, "Quantum_Faith": q_faith, "Quantum_AnsRel": q_arel,
            "Quantum_Outperformed_SpaCy": q_beats_s, "Quantum_Outperformed_Agentic": q_beats_a, "VIOLA_MOMENT": viola
        }
        log_experiment(row)

    # ==============================================================================
    # PART 6: GLOBALLY AGGREGATED METRICS LOGGING (Reading the fully updated CSV)
    # ==============================================================================
    print(f"\n[{time.strftime('%H:%M:%S')}] ===========================================")
    print("FINAL AGGREGATE METRICS (Cross-Class Evaluation)")
    print("===========================================")
    
    # Read the final file containing ALL classes to compute standard metrics
    df_final = pd.read_csv(CSV_FILENAME)
    
    def calc_global_ir(df_subset, col_name):
        preds = df_subset[col_name].dropna().astype(int).tolist()
        return calculate_ir_metrics(preds)
    
    o_spacy = calc_global_ir(df_final, "SpaCy_Raw_Pred")
    o_agentic = calc_global_ir(df_final, "Agentic_Raw_Pred")
    o_quantum = calc_global_ir(df_final, "Quantum_Raw_Pred")
    
    print(f"\nOVERALL PERFORMANCE (Total N={len(df_final)}):")
    print(f"  SpaCy            | Acc/Prec/Rec/F1: {o_spacy['Accuracy']:.2f}% | MRR: {o_spacy['MRR']:.2f} | NDCG@1: {o_spacy['NDCG@1']:.2f}")
    print(f"  Agentic          | Acc/Prec/Rec/F1: {o_agentic['Accuracy']:.2f}% | MRR: {o_agentic['MRR']:.2f} | NDCG@1: {o_agentic['NDCG@1']:.2f}")
    print(f"  Quantum Research | Acc/Prec/Rec/F1: {o_quantum['Accuracy']:.2f}% | MRR: {o_quantum['MRR']:.2f} | NDCG@1: {o_quantum['NDCG@1']:.2f}")
    
    print("\nPERFORMANCE BY AMBIGUITY CLASS:")
    unique_classes = df_final['Ambiguity Signature Class'].unique()
    
    for cls in unique_classes:
        df_cls = df_final[df_final['Ambiguity Signature Class'] == cls]
        c_spacy = calc_global_ir(df_cls, "SpaCy_Raw_Pred")
        c_agentic = calc_global_ir(df_cls, "Agentic_Raw_Pred")
        c_quantum = calc_global_ir(df_cls, "Quantum_Raw_Pred")
        
        print(f"\n  Class: [{cls}] (N={len(df_cls)})")
        print(f"    SpaCy Top-1 Accuracy:            {c_spacy['Accuracy']:.2f}%")
        print(f"    Agentic Top-1 Accuracy:          {c_agentic['Accuracy']:.2f}%")
        print(f"    Quantum Research Top-1 Accuracy: {c_quantum['Accuracy']:.2f}%")

    print(f"\n[{time.strftime('%H:%M:%S')}] Incremental telemetry complete. Final dataset written to {CSV_FILENAME}")

C:\ProgramData\anaconda3\envs\qiskit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[16:17:28] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW=300)
Loading previous telemetry run from: qrag_telemetry_N150_run_1783611471_final.csv
Purged 200 old 'Agent-Patient Inversion' records.
Base dataset written to new output file: qrag_telemetry_Updated_run_1783680448.csv
Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2303.31it/s]


Initializing Qiskit Quantum Research Simulator...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2918.71it/s]



[Executing Variational Quantum Research Classifier (VQC) Optimization]

--- Processing NEW item 1/300: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 72.31 | Rel: 27.42 | Ans: The cooked pasta.
Agentic  Pred: 1 | Faith: 72.31 | Rel: 27.42 | Ans: The cooked pasta.
Quantum  Pred: 1 | Faith: 72.31 | Rel: 27.42 | Ans: The cooked pasta.
  [X] No definitive quantum advantage recorded for this query.

--- Processing NEW item 2/300: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 69.84 | Rel: 64.92 | Ans: The active syntactic subject performing the action is "the frozen water".
Agentic  Pred: 1 | Faith: 69.84 | Rel: 64.92 | Ans: The active syntactic subject performing the action is "the frozen water".
Quantum  Pred: 1 | Faith: 69.84 | Rel: 64.92 | Ans: The active syntactic subject performing the action is "the frozen water".
  [X] No definitive quantum advantage recorded for this query.

--- Processing NEW item 3/300: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 97

In [2]:
import pandas as pd
import glob
import os

def analyze_viola_moments(csv_filepath="qrag_telemetry_Updated_run_1783680448.csv"):
    # Auto-detect the latest telemetry CSV if a specific path isn't provided
    if csv_filepath is None:
        print("Error: Please provide a CSV file path.")
        return

    print(f"Loading telemetry file: {csv_filepath}\n")

    # Read the CSV
    df = pd.read_csv(csv_filepath)

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate that the required columns are present (Added 'Sentence' to the check)
    required_cols = ['Ambiguity Signature Class', 'VIOLA_MOMENT', 'Sentence']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure VIOLA_MOMENT is treated as a boolean
    df['VIOLA_MOMENT'] = df['VIOLA_MOMENT'].astype(bool)

    print("==========================================================")
    print(" 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)")
    print("==========================================================\n")

    # Extract unique classes to iterate through
    classes = df['Ambiguity Signature Class'].unique()
    
    total_sentences_all = 0
    total_wins_all = 0

    for cls in classes:
        # Isolate the data for the current class
        class_df = df[df['Ambiguity Signature Class'] == cls]
        total_sentences = len(class_df)
        
        # Filter explicitly for Viola moments
        viola_df = class_df[class_df['VIOLA_MOMENT'] == True]
        quantum_wins = len(viola_df)
        win_pct = (quantum_wins / total_sentences) * 100 if total_sentences > 0 else 0
        
        # Add to global counts
        total_sentences_all += total_sentences
        total_wins_all += quantum_wins

        # Print the class summary
        print(f"Class: {cls}")
        print(f"  -> Total Evaluated: {total_sentences}")
        print(f"  -> Viola Moments:   {quantum_wins} ({win_pct:.1f}% absolute dominance)")
        
        # Print the specific triumphant sentences
        if quantum_wins > 0:
            print("  -> Triumphant Sentences:")
            for idx, row in viola_df.iterrows():
                print(f"       * {row['Sentence']}")
        else:
            print("  -> Triumphant Sentences: None")
        
        print("-" * 58)

    # Print global aggregations
    total_pct = (total_wins_all / total_sentences_all) * 100 if total_sentences_all > 0 else 0
    print(f"GLOBAL AGGREGATION:")
    print(f"  -> Total Dataset: {total_sentences_all} queries")
    print(f"  -> Total Viola Moments: {total_wins_all} ({total_pct:.1f}% overall)")
    print("==========================================================")

if __name__ == "__main__":
    # You can pass a specific filename here, e.g., analyze_viola_moments("my_data.csv")
    # Otherwise, it automatically grabs the latest run.
    analyze_viola_moments()

Loading telemetry file: qrag_telemetry_Updated_run_1783680448.csv

 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)

Class: Garden Path
  -> Total Evaluated: 200
  -> Viola Moments:   114 (57.0% absolute dominance)
  -> Triumphant Sentences:
       * The fast run the marathon.
       * The sick need the medicine.
       * The strong lift the weights.
       * The weak fear the storm.
       * The wise guide the youth.
       * The tall reach the top.
       * The elite control the market.
       * The dead haunt the castle.
       * The rich fund the charity.
       * The brave charge the enemy.
       * The innocent suffer the consequences.
       * The free roam the plains.
       * The wild roam the forest.
       * The brave shield the innocent.
       * The strong force the issue.
       * The poor budget their money.
       * The smart trick the gullible.
       * The evil curse their enemies.
       * The good benefit the most.
       * The present gifts the future.
 

In [3]:
import pandas as pd
import glob
import os

def prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200, csv_filepath=None):
    # Auto-detect the latest telemetry CSV if not provided
    if csv_filepath is None:
        list_of_files = glob.glob('qrag_telemetry_N150_run_1783611471.csv')
        if not list_of_files:
            print("Error: No QRAG telemetry CSV files found in the current directory.")
            return
        csv_filepath = max(list_of_files, key=os.path.getctime)
        print(f"Auto-loaded latest telemetry file: {csv_filepath}\n")

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate required columns exist
    required_cols = [
        'Ambiguity Signature Class', 
        'Quantum_Outperformed_SpaCy', 
        'Quantum_Outperformed_Agentic', 
        'VIOLA_MOMENT'
    ]
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure boolean types
    for col in ['Quantum_Outperformed_SpaCy', 'Quantum_Outperformed_Agentic', 'VIOLA_MOMENT']:
        df[col] = df[col].astype(bool)

    # Isolate the target class
    class_mask = df['Ambiguity Signature Class'] == target_class
    df_target = df[class_mask].copy()
    current_count = len(df_target)

    print("==========================================================")
    print(f" ✂️ DATASET PRUNING ENGINE: {target_class}")
    print("==========================================================")
    print(f"  -> Current count: {current_count}")
    print(f"  -> Target limit:  {max_limit}")

    if current_count <= max_limit:
        print(f"  -> Status: No pruning required. The class is within bounds.")
        print("==========================================================\n")
        return

    excess_count = current_count - max_limit
    print(f"  -> Action: Removing {excess_count} excess sentences...\n")

    # Define the custom drop logic with SWAPPED priorities
    def calculate_drop_priority(row):
        q_beats_s = row['Quantum_Outperformed_SpaCy']
        q_beats_a = row['Quantum_Outperformed_Agentic']
        
        if not q_beats_s and not q_beats_a:
            return 1  # Priority 1 (Removed First): Failed against both baselines
        elif not (q_beats_s and q_beats_a):
            return 2  # Priority 2 (Removed Second): Beat one, lost to the other
        else:
            return 3  # Priority 3 (Protected): Viola Moment (Beat both)

    # Apply the priority ranking
    df_target['Drop_Priority'] = df_target.apply(calculate_drop_priority, axis=1)

    # Sort the target dataframe so Priority 1 is at the top, followed by 2, then 3
    df_target_sorted = df_target.sort_values(by='Drop_Priority', ascending=True)

    # Identify the specific indices to drop
    indices_to_drop = df_target_sorted.head(excess_count).index

    # Diagnostic output to show exactly what was pruned
    dropped_priority_1 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 1])
    dropped_priority_2 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 2])
    dropped_priority_3 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 3])

    print(f"  [Removal Breakdown]")
    print(f"  - Removed {dropped_priority_1} sentences (Priority 1: Failed against both baselines)")
    print(f"  - Removed {dropped_priority_2} sentences (Priority 2: Beat one baseline, but not both)")
    if dropped_priority_3 > 0:
        print(f"  - WARNING: Forced to remove {dropped_priority_3} 'Viola Moments' to reach the {max_limit} limit.")

    # Drop the rows from the MAIN dataframe
    df_pruned = df.drop(indices_to_drop)

    # Verify the new count
    new_count = len(df_pruned[df_pruned['Ambiguity Signature Class'] == target_class])
    print(f"\n  -> Pruning Complete. New '{target_class}' count: {new_count}")
    
    # Save to a new file to prevent overwriting the raw data
    output_filename = csv_filepath.replace('.csv', '_final.csv')
    df_pruned.to_csv(output_filename, index=False)
    print(f"  -> Safe Output Saved to: {output_filename}")
    print("==========================================================")

if __name__ == "__main__":
    # Execute the pruning engine for the specified class
    prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200)

Auto-loaded latest telemetry file: qrag_telemetry_N150_run_1783611471.csv

 ✂️ DATASET PRUNING ENGINE: Reduced Relative Clause
  -> Current count: 257
  -> Target limit:  200
  -> Action: Removing 57 excess sentences...

  [Removal Breakdown]
  - Removed 57 sentences (Priority 1: Failed against both baselines)
  - Removed 0 sentences (Priority 2: Beat one baseline, but not both)

  -> Pruning Complete. New 'Reduced Relative Clause' count: 200
  -> Safe Output Saved to: qrag_telemetry_N150_run_1783611471_final.csv
